In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
# =========================================================
# 1. STATE
# =========================================================
class State(TypedDict):
    count: int
# =========================================================
# 2. NODES
# =========================================================
def add_one(state: State):
    print("ADD_ONE running...")
    return {
        "count": state["count"] + 1
    }
def add_two(state: State):
    print("ADD_TWO running...")
    return {
        "count": state["count"] + 2
    }
def add_three(state: State):
    print("ADD_THREE running...")
    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)
builder.add_node("add_three", add_three)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", "add_three")
builder.add_edge("add_three", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. ORIGINAL THREAD
# =========================================================

original_config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n========================================")
print("ORIGINAL EXECUTION")
print("========================================")

original_result = graph.invoke(
    {"count": 0},
    config=original_config
)

print(
    "Original Result:",
    original_result
)


# =========================================================
# 6. GET ORIGINAL HISTORY
# =========================================================

history = list(
    graph.get_state_history(
        original_config
    )
)

print("\n========================================")
print("CHECKPOINT HISTORY")
print("========================================")

for index, state in enumerate(history):

    print(
        f"\nCheckpoint {index}"
    )

    print(
        "Checkpoint ID:",
        state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 7. SELECT A PAST CHECKPOINT
# =========================================================

# History is newest -> oldest.
#
# We select a checkpoint where
# execution has not finished yet.

target_state = None

for state in history:

    if state.next:
        target_state = state
        break


if target_state is None:

    print(
        "\nNo suitable checkpoint found."
    )

else:

    old_checkpoint_id = (
        target_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    # =====================================================
    # 8. ORIGINAL HISTORICAL CONFIG
    # =====================================================

    old_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": old_checkpoint_id
        }
    }


    old_state = graph.get_state(
        old_config
    )


    print("\n========================================")
    print("OLD CHECKPOINT")
    print("========================================")

    print(
        "Checkpoint:",
        old_checkpoint_id
    )

    print(
        "State:",
        old_state.values
    )

    print(
        "Next:",
        old_state.next
    )


    # =====================================================
    # 9. CREATE NEW FORK THREAD
    # =====================================================

    fork_config = {
        "configurable": {
            "thread_id": "thread-002"
        }
    }


    # =====================================================
    # 10. CREATE FORK FROM OLD STATE
    # =====================================================

    fork_initial_state = {
        "count": old_state.values["count"]
    }


    fork_result = graph.invoke(
        fork_initial_state,
        config=fork_config
    )


    print("\n========================================")
    print("INITIAL FORK")
    print("========================================")

    print(
        "Fork Result:",
        fork_result
    )


    # =====================================================
    # 11. GET FORK HISTORY
    # =====================================================

    fork_history = list(
        graph.get_state_history(
            fork_config
        )
    )


    # =====================================================
    # 12. SELECT FORK CHECKPOINT
    # =====================================================

    fork_target = None

    for state in fork_history:

        if state.next:
            fork_target = state
            break


    if fork_target is None:

        print(
            "\nNo fork checkpoint available."
        )

    else:

        fork_checkpoint_id = (
            fork_target.config[
                "configurable"
            ]["checkpoint_id"]
        )


        fork_checkpoint_config = {
            "configurable": {
                "thread_id": "thread-002",
                "checkpoint_id": fork_checkpoint_id
            }
        }


        # =================================================
        # 13. MODIFY FORKED STATE
        # =================================================

        print("\n========================================")
        print("MODIFY FORKED STATE")
        print("========================================")

        print(
            "Before:",
            fork_target.values
        )


        # IMPORTANT:
        # Only the FORKED thread is modified.

        graph.update_state(
            fork_checkpoint_config,
            {
                "count": 10
            }
        )


        modified_state = graph.get_state(
            fork_checkpoint_config
        )


        print(
            "After:",
            modified_state.values
        )


        # =================================================
        # 14. CONTINUE FORKED EXECUTION
        # =================================================

        print("\n========================================")
        print("CONTINUE FORKED EXECUTION")
        print("========================================")

        modified_result = graph.invoke(
            None,
            config=fork_checkpoint_config
        )


        print(
            "Modified Fork Result:",
            modified_result
        )


        # =================================================
        # 15. ORIGINAL THREAD CHECK
        # =================================================

        original_state = graph.get_state(
            original_config
        )


        print("\n========================================")
        print("ORIGINAL THREAD")
        print("========================================")

        print(
            "Thread:",
            original_config[
                "configurable"
            ]["thread_id"]
        )

        print(
            "State:",
            original_state.values
        )


        # =================================================
        # 16. FORK THREAD CHECK
        # =================================================

        final_fork_state = graph.get_state(
            fork_config
        )


        print("\n========================================")
        print("FORK THREAD")
        print("========================================")

        print(
            "Thread:",
            fork_config[
                "configurable"
            ]["thread_id"]
        )

        print(
            "State:",
            final_fork_state.values
        )


        # =================================================
        # 17. FINAL COMPARISON
        # =================================================

        print("\n========================================")
        print("FINAL COMPARISON")
        print("========================================")

        print(
            "Original:",
            original_state.values
        )

        print(
            "Forked:",
            final_fork_state.values
        )


ORIGINAL EXECUTION
ADD_ONE running...
ADD_TWO running...
ADD_THREE running...
Original Result: {'count': 6}

CHECKPOINT HISTORY

Checkpoint 0
Checkpoint ID: 1f1bd020-6d60-69f2-8003-bf7a78f25881
Values: {'count': 6}
Next: ()

Checkpoint 1
Checkpoint ID: 1f1bd020-6d5d-64b7-8002-8936e33827fe
Values: {'count': 3}
Next: ('add_three',)

Checkpoint 2
Checkpoint ID: 1f1bd020-6d57-626f-8001-06cd7c08933a
Values: {'count': 1}
Next: ('add_two',)

Checkpoint 3
Checkpoint ID: 1f1bd020-6d50-6dbd-8000-5b589c8ed3b9
Values: {'count': 0}
Next: ('add_one',)

Checkpoint 4
Checkpoint ID: 1f1bd020-6d47-6f90-bfff-0466137b4721
Values: {}
Next: ('__start__',)

OLD CHECKPOINT
Checkpoint: 1f1bd020-6d5d-64b7-8002-8936e33827fe
State: {'count': 3}
Next: ('add_three',)
ADD_ONE running...
ADD_TWO running...
ADD_THREE running...

INITIAL FORK
Fork Result: {'count': 9}

MODIFY FORKED STATE
Before: {'count': 6}


KeyError: 'checkpoint_ns'